In [1]:
library(Seurat)
library(SeuratDisk)
library(dplyr)
library(ggplot2)

data_dir <- '/singleCell/Ruzicka_snRNA_Seq/data/' # Ruzicka 
data_dir1 <- '/Aging/Data/'
out_dir <- '/Aging/Aging_Clock/Output/'
plot_dir <- '/Aging/Aging_Clock/Plots/'
setwd(plot_dir)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:base’:

    intersect


Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
# Ruzicka Dataset -------------------------------------

obj <- readRDS(paste0(data_dir,'22_08_12_seurat_raw_FILTERED_neurons_only_controls_only_lognormalized_scaled.rds'))
obj1 <- readRDS(paste0(data_dir1,"Ruzicka_snRNA_Seq/22_08_12_seurat_raw_FILTERED_neurons_only_controls_only_lognormalized_AUCell_M0_M9_M24_dd.rds"))

pred_age <- read.csv(paste0(out_dir,'24_10_02_model_M24_predictions_gene_filtering_mean_age_corrected_pred_age.csv'),row.names = 1)
pred_age <- pred_age[,which(names(pred_age) %in% c('Id','predicted_age_0','predicted_age_24'))]  

metadata1 = obj1@meta.data[,c(4,36:58)]

metadata = obj@meta.data[,4,drop = FALSE]
metadata = left_join(metadata, pred_age, by = 'Id')                             
rownames(metadata) = metadata$Id
obj = AddMetaData(obj, metadata)

Idents(obj) <- "age_status"
levels(obj) = c('young','old')
obj$age_status <- factor(x = obj$age_status, levels = c('young', 'old'))

# Add Mitochondrial genes 
#obj[["percent.mt"]] <- PercentageFeatureSet(object = obj, pattern = "^MT")

In [20]:
# Manuscript Figure 3A --------------------------------------------------------
#------------------------------------------------------------------------------

# Violin Plots

pdf('VlnPlot_Ruzicka_pred_age.pdf', width=9,height=8)
p <- VlnPlot(obj, 'predicted_age_0', group.by = 'Age', pt.size=0) 
p + 
  scale_fill_manual(values = rep('#21FE80', length(unique(obj$Age)))) + # Same fill color for all violins
  geom_violin(color = '#000435', size = 0.5) + # Set outline color and size
  theme_minimal() + # Optional: adjust the theme 
  NoLegend()


p <- VlnPlot(obj, 'predicted_age_24', group.by = 'Age', pt.size=0) 
p + 
  scale_fill_manual(values = rep('#21FE80', length(unique(obj$Age)))) + # Same fill color for all violins
  geom_violin(color = '#000435', size = 0.5) + # Set outline color and size
  theme_minimal() + # Optional: adjust the theme 
  NoLegend()

dev.off()

Warning message:
“Removing 26758 cells missing data for vars requested”
Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=FALSE`

Warning message:
“Removed 26520 rows containing non-finite outside the scale range
(`stat_ydensity()`).”
Warning message:
“Removed 26520 rows containing non-finite outside the scale range
(`stat_ydensity()`).”
Warning message:
“Removing 26758 cells missing data for vars requested”
Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=FALSE`

Warning message:
“Removed 26520 rows containing non-finite outside the scale range
(`stat_ydensity()`).”
Warning message:
“Removed 26520 rows containing non-finite outside the scale range
(`stat_ydensity()`).”


pdf 
  2

In [51]:
# Manuscript Figure 4 ---------------------------------------------------------
#------------------------------------------------------------------------------

custom_palette <- c(
"#9ec6ba", "#a4ebc5", "#b5b477", "#aecdc4", "#825ca9",
"#9c68a7", "#7f7f7f", "#6e6e6d", "#555555", "#bfa0a7",
"#eec3e0", "#9ec6ad", "#689e73", "#757575", "#6e6e6d",
"#9c9c64", "#9c6e6d", "#6e6e6d", "#9c9c64", "#a4ebbb",
"#9ec6ba"
)
Idents(obj) <- 'Celltype'
levels(obj) <- c('Ex-L2','Ex-L23','Ex-L3','Ex-L4_MYLK','Ex-L45_LRRK1','Ex-L45_MET','Ex-L56','Ex-L56_CC_NTNG2','Ex-L5b_HTR2C','Ex-L5b_VEN','Ex-L6_CC_SEMA3A','Ex-L6b_SEMA3D','Ex-L6b_SEMA3E','Ex-NRGN',
       'In-PV_Basket','In-PV_Chandelier','In-Reelin','In-Rosehip_CHST9','In-Rosehip_TRPC3','In-SST','In-VIP')
pdf('DimPlot_Ruzicka_neurons.pdf', width=6,height=8)
DimPlot(obj, cols = custom_palette, label = T) + NoLegend()
dev.off()

pdf('Featureplot_Ruzicka_geneset_sc.pdf', width=15,height=10)
FeaturePlot(obj, 'geneset_sc0', split.by = 'age_status', cols = c('#000435','#FFCBA4','#21FE80'), keep.scale='all') & theme(legend.position = "right")
FeaturePlot(obj, 'geneset_sc24',split.by = 'age_status', cols = c('#000435','#FFCBA4','#21FE80'),keep.scale='all') & theme(legend.position = "right")

FeaturePlot(obj, 'predicted_age_0', split.by = 'age_status', cols = c('#000435','#FFCBA4','#21FE80'), keep.scale='all') & theme(legend.position = "right")
FeaturePlot(obj, 'predicted_age_24', split.by = 'age_status', cols = c('#000435','#FFCBA4','#21FE80'),keep.scale='all') & theme(legend.position = "right")

dev.off()

pdf('Featureplot_Ruzicka_FKBP5_SST.pdf', width=7,height=7)
FeaturePlot(obj, c('FKBP5', 'SST'), split.by = 'age_status', keep.scale='feature', 
             pt.size = 1.5) & theme(legend.position = "right")
dev.off()

Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=FALSE`



pdf 
  2

Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=FALSE`

Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=FALSE`

Warning message:
“Removed 70708 rows containing missing values or values outside the scale range
(`geom_scattermore()`).”
Warning message:
“Removed 70708 rows containing missing values or values outside the scale range
(`geom_scattermore()`).”
Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=FALSE`

Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=FALSE`

Warning message:
“Removed 70708 rows containing missing values or values outside the scale range
(`geom_scattermore()`).”
Warning message:
“Removed 70708 rows containing missing values or values outside the scale range
(`geom_scattermore()`).”
Rasterizing points since number of points exceeds 100,000.
To disable this behavior set `raster=

pdf 
  2

In [2]:
# Anderson Dataset ------------------------------------

obj <- readRDS(paste0(data_dir1,"Anderson/snRNA_Seq/GSE214979_filtered_feature_bc_matrix_processed_seurat_neurons_only_with_Ruzicka_AUCell_M0_M9_M24_dd.rds"))
rownames(obj@meta.data) <- obj@meta.data$barcode # make sure rownames are cell barcodes 
obj <- RunUMAP(obj,dims = 1:10)

Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”
10:04:53 UMAP embedding parameters a = 0.9922 b = 1.112

10:04:53 Read 56215 rows and found 10 numeric columns

10:04:53 Using Annoy for neighbor search, n_neighbors = 30

10:04:53 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

10:05:01 Writing NN index file to temp file /tmp/RtmpZ4H5Y3/file1f30e67e0480d

10:05:01 Searching Annoy index using 1 thread, search_k = 3000

10:05:30 Annoy recall = 100%

10:05:31 Commencing smooth kNN distance calibration using 1 thread
 with target n_neighbors

In [53]:
# Manuscript Figure 4 ---------------------------------------------------------
#------------------------------------------------------------------------------

custom_palette <- c(
"#9ec6ba", "#a4ebc5", "#b5b477", "#aecdc4", "#825ca9",
"#9c68a7", "#7f7f7f", "#6e6e6d", "#555555", "#bfa0a7",
"#eec3e0", "#9ec6ad", "#689e73", "#757575", "#6e6e6d",
"#9c9c64", "#9c6e6d", "#6e6e6d", "#9c9c64", "#a4ebbb",
"#9ec6ba"
)

Idents(obj) <- 'predicted.id'
levels(obj) <- c('Ex-L2','Ex-L23','Ex-L3','Ex-L4_MYLK','Ex-L45_LRRK1','Ex-L45_MET','Ex-L56','Ex-L56_CC_NTNG2','Ex-L5b_HTR2C','Ex-L5b_VEN','Ex-L6_CC_SEMA3A','Ex-L6b_SEMA3D','Ex-L6b_SEMA3E','Ex-NRGN',
       'In-PV_Basket','In-PV_Chandelier','In-Reelin','In-Rosehip_CHST9','In-Rosehip_TRPC3','In-SST','In-VIP')

pdf('DimPlot_Anderson_neurons.pdf', width=6,height=8)
DimPlot(obj, cols = custom_palette, label = T) + NoLegend()
dev.off()


# If first value in feature column is NA
rownames(obj@meta.data) <- obj@meta.data$barcode # make sure rownames are cell barcodes 
new.cell.order <- rownames(obj@meta.data)[order(obj@meta.data$geneset_sc0)] # order cells to avoid first value to be NA
Idents(obj) <- "Status"
levels(obj) = c('Ctrl','AD')
obj$Status <- factor(x = obj$Status, levels = c('Ctrl','AD'))

pdf('Featureplot_Anderson_geneset_sc.pdf', width=15,height=10)
FeaturePlot(obj, 'geneset_sc0', split.by = 'Status', cols = c('#000435','#FFCBA4','#21FE80'), keep.scale='all',cells = new.cell.order) & theme(legend.position = "right")
FeaturePlot(obj, 'geneset_sc24',split.by = 'Status', cols = c('#000435','#FFCBA4','#21FE80'),keep.scale='all',cells = new.cell.order) & theme(legend.position = "right")

dev.off()

pdf 
  2

pdf 
  2